# RetailMind AI — Day 4: Feature Engineering & Ensemble Learning

This notebook continues the Day 1–3 Telco Customer Churn project.

**Tasks:** engineer domain features, select useful features, train Random Forest and XGBoost, compare with the Day 3 champion, and save the final production pipeline.

Run from the `notebooks/` folder or project root. The notebook detects the project root automatically.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Put the CSV in data/.")


## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
display(df["Churn"].value_counts(normalize=True).rename("proportion"))


## 3. Prepare features and target

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
X = df.drop(columns=["Churn", "customerID"]).copy()
y = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train rows:", len(X_train), "| Test rows:", len(X_test))


## 4. Feature engineering

New features include tenure groups, average monthly spend, charge ratio, contract flags, service count, and service/billing flags. The transformer is included in the saved pipeline so future prediction data receives the same transformations.

In [ ]:
class RetailFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["TotalCharges"] = pd.to_numeric(X["TotalCharges"], errors="coerce")

        X["TenureGroup"] = pd.cut(
            X["tenure"], bins=[-1, 12, 24, 48, np.inf],
            labels=["New", "Medium", "Long", "Loyal"]
        ).astype("object")

        safe_tenure = X["tenure"].replace(0, np.nan)
        X["AverageMonthlySpend"] = X["TotalCharges"] / safe_tenure
        X["ChargeRatio"] = X["MonthlyCharges"] / (X["TotalCharges"] + 1)

        X["IsMonthToMonth"] = (X["Contract"] == "Month-to-month").astype(int)
        X["IsOneYearContract"] = (X["Contract"] == "One year").astype(int)
        X["IsTwoYearContract"] = (X["Contract"] == "Two year").astype(int)

        service_columns = [
            "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
        ]
        X["ServiceCount"] = sum((X[col] == "Yes").astype(int) for col in service_columns)
        X["HasInternet"] = (X["InternetService"] != "No").astype(int)
        X["HasPhone"] = (X["PhoneService"] == "Yes").astype(int)
        X["PaperlessBillingFlag"] = (X["PaperlessBilling"] == "Yes").astype(int)
        return X


## 5. Preprocessing and feature selection

In [ ]:
feature_engineer = RetailFeatureEngineer()
X_train_eng = feature_engineer.fit_transform(X_train)
X_test_eng = feature_engineer.transform(X_test)

numeric_features = X_train_eng.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train_eng.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Determine a safe feature count using training data only.
encoded_train = preprocessor.fit_transform(X_train_eng, y_train)
k_features = min(40, encoded_train.shape[1])
print("Encoded feature count:", encoded_train.shape[1])
print("Selected feature count:", k_features)


## 6. Define ensemble models

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

candidate_models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_split=5,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight, eval_metric="logloss",
        random_state=42, n_jobs=-1
    )
}


## 7. Train and evaluate Day 4 models

Five-fold cross-validation recall is measured on the training split. The held-out test split is used for final evaluation.

In [ ]:
results = []
trained_models = {}

for model_name, classifier in candidate_models.items():
    print(f"Training {model_name}...")

    model_pipeline = Pipeline([
        ("feature_engineering", RetailFeatureEngineer()),
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=k_features)),
        ("classifier", classifier)
    ])

    cv_scores = cross_val_score(
        model_pipeline, X_train, y_train,
        cv=5, scoring="recall", n_jobs=1
    )

    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    y_prob = model_pipeline.predict_proba(X_test)[:, 1]
    trained_models[model_name] = model_pipeline

    results.append({
        "Model": model_name,
        "CV Recall": cv_scores.mean(),
        "CV Recall Std": cv_scores.std(),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    print(f"CV recall: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")


## 8. Compare with the Day 3 champion (if available)

Expected filename: `models/churn_v3.joblib`. Change the path below if your Day 3 model has a different filename.

In [ ]:
day3_model_path = MODELS_DIR / "churn_v3.joblib"

if day3_model_path.exists():
    day3_model = joblib.load(day3_model_path)
    day3_cv = cross_val_score(
        day3_model, X_train, y_train, cv=5, scoring="recall", n_jobs=1
    )
    day3_pred = day3_model.predict(X_test)
    day3_prob = day3_model.predict_proba(X_test)[:, 1]

    trained_models["Day 3 Champion"] = day3_model
    results.append({
        "Model": "Day 3 Champion",
        "CV Recall": day3_cv.mean(),
        "CV Recall Std": day3_cv.std(),
        "Accuracy": accuracy_score(y_test, day3_pred),
        "Precision": precision_score(y_test, day3_pred, zero_division=0),
        "Recall": recall_score(y_test, day3_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, day3_prob)
    })
    print("Day 3 champion included in comparison.")
else:
    print("Day 3 model not found at:", day3_model_path)
    print("Comparing Day 4 models only.")


## 9. Compare models and select champion

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by=["CV Recall", "ROC-AUC"], ascending=False
).reset_index(drop=True)

display(results_df)
results_df.to_csv(REPORTS_DIR / "day4_model_comparison.csv", index=False)

champion_name = results_df.loc[0, "Model"]
champion_model = trained_models[champion_name]
print("Selected champion:", champion_name)


## 10. Save production model and champion report

In [ ]:
production_path = MODELS_DIR / "churn_v4.joblib"
joblib.dump(champion_model, production_path)

with open(REPORTS_DIR / "day4_champion.txt", "w", encoding="utf-8") as report:
    report.write(f"Champion Model: {champion_name}\n")
    report.write(f"CV Recall: {results_df.loc[0, 'CV Recall']:.4f}\n")
    report.write(f"Test Recall: {results_df.loc[0, 'Recall']:.4f}\n")
    report.write(f"Test ROC-AUC: {results_df.loc[0, 'ROC-AUC']:.4f}\n")

print("Saved:", production_path)
print("Saved:", REPORTS_DIR / "day4_champion.txt")


## 11. Visualize comparison

In [ ]:
ax = results_df.set_index("Model")[["CV Recall", "Recall", "ROC-AUC"]].plot(
    kind="bar", figsize=(10, 5)
)
ax.set_title("Day 4 Model Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_xlabel("Model")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plot_path = REPORTS_DIR / "day4_model_comparison.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print("Saved plot:", plot_path)


## 12. Final checks and submission

Generated files:
- `models/churn_v4.joblib`
- `reports/day4_model_comparison.csv`
- `reports/day4_champion.txt`
- `reports/day4_model_comparison.png`

Run all cells successfully, enter the actual metric values in README.md, then commit and push the project.

```bash
git status
git add .
git commit -m "Complete Day 4 feature engineering and ensemble models"
git push origin main
```

Do not commit virtual environments, credentials, or any dataset you are not permitted to redistribute.